# Budget sweep on a free GPU runtime

This notebook is a runnable entry point for `scripts/run_sweep.py`. It reconstructs the
*configuration* of the recorded sweep in `results/person1/sweep_annealed.json`; it is not
the original Kaggle kernel, which was not kept in the repository. The important difference
is that `max_updates` is now derived from the batch artifact and recorded in every run's
`metadata.json` instead of living only in prose.


In [ ]:
import sys
import torch

print(sys.version)
print(torch.__version__, torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable a GPU accelerator before running this notebook.'


In [ ]:
# Install by name. Do not install requirements.txt here: it pins a CPU torch wheel.
!pip install -q 'transformers>=4.40.0' 'datasets>=2.18.0' 'scikit-learn>=1.4.0' 'scipy>=1.12.0' 'numpy>=1.26.0'

In [ ]:
!git clone --depth 1 --branch main https://github.com/dakshadev/7th-sem.git
%cd 7th-sem
!git rev-parse HEAD

In [ ]:
# 0.69M + 5M, C/A/B, two seeds. The schedule is derived per budget and recorded.
!python scripts/run_sweep.py \
  --output /kaggle/working/sweep \
  --out /kaggle/working/sweep.json \
  --budgets 700000,5000000 \
  --variants mlm,latent_a,latent_b \
  --seeds 17,43 \
  --max-records 90000 \
  --batch-size 8 \
  --device cuda \
  --probe --random-baseline

## Recording the result

Copy `sweep.json` into `results/` only after checking that every row's `max_updates` equals
`round(budget / tokens_per_update)`, that the run's `metadata.json` records the same value,
and that no run reports `stop_reason: failed`. Keep the JSON and the artifact manifest;
do not commit checkpoints, batch tensors or embeddings.
